In [2]:
import os
import re
from datetime import datetime, timedelta
from pathlib import Path

def standardize_and_find_missing_mondays(folder_path_str):
    """
    1. Renames VNM/IEX files to 'YYYY-MM-DD.xlsx'.
    2. (NEW) Fixes 'YYYY_MM_DD.xlsx' files to 'YYYY-MM-DD.xlsx'.
    3. Scans all 'YYYY-MM-DD.xlsx' files to find the date range.
    4. Reports any missing Mondays within that range.
    """
    
    folder = Path(folder_path_str)

    # --- REGEX 1: Matches VNM_... or IEX_... files ---
    input_pattern = re.compile(
        r"^(?:VNM|IEX)_(\d{1,2})_([A-Za-z]+)_(?:.*?)(\d{4})(\.xlsx)$", 
        re.IGNORECASE
    )
    
    # --- REGEX 2: Matches 'YYYY-MM-DD.xlsx' (Target format) ---
    date_pattern = re.compile(
        r"^(\d{4})-(\d{2})-(\d{2})(\.xlsx)$"
    )
    
    # --- REGEX 3 (MỚI): Matches 'YYYY_MM_DD.xlsx' (Format cần sửa) ---
    underscore_pattern = re.compile(
        r"^(\d{4})_(\d{2})_(\d{2})(\.xlsx)$"
    )

    if not folder.is_dir():
        print(f"--- LỖI ---")
        print(f"Không tìm thấy thư mục tại đường dẫn:")
        print(f"{folder}")
        return

    print(f"--- BẮT ĐẦU QUÁ TRÌNH CHUẨN HÓA VÀ KIỂM TRA ---")
    print(f"Quét thư mục: {folder}\n")
    
    renamed_count = 0  # Đếm VNM/IEX
    fixed_count = 0    # (MỚI) Đếm Y_M_D
    existing_count = 0 # Đếm Y-M-D
    skipped_count = 0
    
    # List to store all dates (as date objects) we find
    existing_dates = []

    # --- PART 1: Rename/Fix files and collect all dates ---
    for file_path in folder.glob("*.xlsx"):
        file_name = file_path.name
        
        match_input = input_pattern.match(file_name)
        match_underscore = underscore_pattern.match(file_name) # (MỚI)
        match_date = date_pattern.match(file_name)

        if match_input:
            # Đây là tệp VNM/IEX, cần đổi tên
            day_str, month_str, year_str, extension = match_input.groups()
            
            try:
                start_date_str = f"{day_str} {month_str} {year_str}"
                start_date_obj = None
                try:
                    start_date_obj = datetime.strptime(start_date_str, "%d %b %Y")
                except ValueError:
                    start_date_obj = datetime.strptime(start_date_str, "%d %B %Y")

                new_file_name = start_date_obj.strftime("%Y-%m-%d") + extension
                new_file_path = folder / new_file_name
                
                file_path.rename(new_file_path)
                
                print(f"ĐỔI TÊN: '{file_name}' -> '{new_file_name}'")
                existing_dates.append(start_date_obj.date())
                renamed_count += 1
            
            except Exception as e:
                print(f"BỎ QUA (Lỗi đổi tên): Không thể xử lý '{file_name}'. Lỗi: {e}")
                skipped_count += 1

        elif match_underscore:
            # (MỚI) Đây là tệp YYYY_MM_DD, cần sửa
            try:
                year, month, day, extension = match_underscore.groups()
                
                # Tạo tên mới với gạch nối
                new_file_name = f"{year}-{month}-{day}{extension}"
                new_file_path = folder / new_file_name
                
                file_path.rename(new_file_path)
                
                print(f"ĐÃ SỬA:   '{file_name}' -> '{new_file_name}'")
                
                # Thêm ngày này vào danh sách để kiểm tra
                date_obj = datetime(int(year), int(month), int(day)).date()
                existing_dates.append(date_obj)
                fixed_count += 1
                
            except Exception as e:
                print(f"BỎ QUA (Lỗi sửa): Không thể xử lý '{file_name}'. Lỗi: {e}")
                skipped_count += 1
                
        elif match_date:
            # Đây là tệp YYYY-MM-DD đã đúng định dạng
            try:
                year, month, day = [int(g) for g in match_date.groups()[:3]]
                date_obj = datetime(year, month, day).date()
                existing_dates.append(date_obj)
                existing_count += 1
            except Exception as e:
                print(f"BỎ QUA (Lỗi ngày): '{file_name}' có vẻ là ngày không hợp lệ. Lỗi: {e}")
                skipped_count += 1
                
        else:
            # Tệp không liên quan
            # print(f"BỎ QUA: Tệp không liên quan '{file_name}'") # Bật nếu bạn muốn xem
            skipped_count += 1

    print("\n--- HOÀN TẤT CHUẨN HÓA TÊN ---")
    print(f"Tổng số tệp đã đổi tên (VNM/IEX): {renamed_count}")
    print(f"Tổng số tệp đã sửa (Y_M_D):    {fixed_count}") # (MỚI)
    print(f"Tổng số tệp đã tồn tại (Y-M-D): {existing_count}")
    print(f"Tổng số tệp đã bỏ qua:         {skipped_count}")

    # --- PART 2: Check for missing Mondays ---
    if not existing_dates:
        print("\n--- KHÔNG TÌM THẤY TỆP NGÀY NÀO ĐỂ KIỂM TRA ---")
        return

    min_date = min(existing_dates)
    max_date = max(existing_dates)
    
    print(f"\n--- KIỂM TRA CÁC NGÀY THỨ HAI BỊ THIẾU ---")
    print(f"Phạm vi kiểm tra: Từ {min_date.isoformat()} đến {max_date.isoformat()}")

    existing_dates_set = set(existing_dates)
    missing_mondays = []
    
    current_date = min_date
    one_week = timedelta(days=7)

    while current_date <= max_date:
        if current_date not in existing_dates_set:
            missing_mondays.append(current_date)
            
        current_date += one_week

    # --- PART 3: Report results ---
    if not missing_mondays:
        print("\n>>> TUYỆT VỜI: Không tìm thấy ngày Thứ Hai nào bị thiếu trong chuỗi.")
    else:
        print(f"\n>>> CẢNH BÁO: Phát hiện {len(missing_mondays)} báo cáo Thứ Hai bị thiếu:")
        for date in missing_mondays:
            print(f"  - {date.isoformat()}.xlsx")

# ===================================================================
# --- (MAIN) EXECUTION ---
# ===================================================================

# Define the target folder path provided
folder_to_process = r"C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\Rawdata\STORAGE_OUTPUT_RTA"

# Run the new function
standardize_and_find_missing_mondays(folder_to_process)

--- BẮT ĐẦU QUÁ TRÌNH CHUẨN HÓA VÀ KIỂM TRA ---
Quét thư mục: C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\Rawdata\STORAGE_OUTPUT_RTA


--- HOÀN TẤT CHUẨN HÓA TÊN ---
Tổng số tệp đã đổi tên (VNM/IEX): 0
Tổng số tệp đã sửa (Y_M_D):    0
Tổng số tệp đã tồn tại (Y-M-D): 87
Tổng số tệp đã bỏ qua:         1

--- KIỂM TRA CÁC NGÀY THỨ HAI BỊ THIẾU ---
Phạm vi kiểm tra: Từ 2024-01-01 đến 2025-11-17

>>> CẢNH BÁO: Phát hiện 12 báo cáo Thứ Hai bị thiếu:
  - 2024-01-29.xlsx
  - 2024-02-19.xlsx
  - 2024-02-26.xlsx
  - 2024-03-04.xlsx
  - 2024-03-11.xlsx
  - 2024-03-18.xlsx
  - 2024-03-25.xlsx
  - 2024-04-01.xlsx
  - 2024-04-08.xlsx
  - 2024-04-15.xlsx
  - 2024-04-22.xlsx
  - 2024-12-02.xlsx


In [2]:
import polars as pl
from pathlib import Path

def convert_xlsx_to_csv(directory_path: str) -> None:
    target_dir = Path(directory_path)
    
    for excel_file in target_dir.glob("*.xlsx"):
        print(f"Processing file: {excel_file.name}")
        
        csv_file_path = excel_file.with_suffix(".csv")
        
        df = pl.read_excel(excel_file)
        df.write_csv(csv_file_path)
        
        print(f"Successfully converted to: {csv_file_path.name}")
        print("-" * 50)

if __name__ == "__main__":
    target_directory = r"C:\Users\ADMIN\Concentrix Corporation\WFM-Expedia-HCM - Branding files\Rawdata\OUTPUT_AGENT_IEX_INTERVALS"
    
    print("Starting conversion process...")
    convert_xlsx_to_csv(target_directory)
    print("All files have been converted completely.")

Starting conversion process...
Processing file: 2025-05-12.xlsx
Successfully converted to: 2025-05-12.csv
--------------------------------------------------
Processing file: 2025-05-19.xlsx
Successfully converted to: 2025-05-19.csv
--------------------------------------------------
Processing file: 2025-05-26.xlsx
Successfully converted to: 2025-05-26.csv
--------------------------------------------------
Processing file: 2025-06-02.xlsx
Successfully converted to: 2025-06-02.csv
--------------------------------------------------
Processing file: 2025-06-09.xlsx
Successfully converted to: 2025-06-09.csv
--------------------------------------------------
Processing file: 2025-06-16.xlsx
Successfully converted to: 2025-06-16.csv
--------------------------------------------------
Processing file: 2025-06-23.xlsx
Successfully converted to: 2025-06-23.csv
--------------------------------------------------
Processing file: 2025-09-01.xlsx
Successfully converted to: 2025-09-01.csv
------------